In [1]:
# set wd
import os
from pathlib import Path
print(os.getcwd())
wd = Path(os.getcwd())
if wd.name == "notebooks":
    %cd ..
print(f"Working Dir Base: {(os.getcwd())}")

/Users/peli/Projects/Repositories/MEGPypes/notebooks
/Users/peli/Projects/Repositories/MEGPypes
Working Dir Base: /Users/peli/Projects/Repositories/MEGPypes


In [2]:
import yaml
import time
from bids.layout import BIDSLayout
# import
from megpypes.pipelines.meg_preprocessing import create_meg_preprocessing


/Users/peli/Projects/Repositories/MEGPypes/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loaded a MEG dataset from here:
https://openneuro.org/datasets/ds006629/versions/1.0.1

using datalad
```
datalad install -s https://github.com/OpenNeuroDatasets/ds006629.git data/ds006629
```
```
cd output/ds006629
datalad get -r .
```


First we need to grab data from our dataset.
We assume the data operate within a bids compliant data structure
https://peerherholz.github.io/workshop_weizmann/nipype/notebooks/basic_data_input_bids.html



In [3]:
# do some data inspection using pybids?
layout = BIDSLayout("data/ds006629/")
print(layout)
# subjects
subjects = layout.get_subjects()
print(f"Subjects: {subjects}")
# sessions
sessions = layout.get_sessions()
print(f"Sessions: {sessions}")
# task
layout.get_tasks()
# datatypes
bidstypes = layout.get_datatypes()
print(f"Data types: {bidstypes}")
# suffixes
print(layout.get_suffixes(datatype='func'))
# see tasks
layout.get_tasks()
print(f"Tasks: {layout.get_tasks()}")
# see dataset description
layout.get_dataset_description()
#

# see data metadata


BIDS Layout: ...itories/MEGPypes/data/ds006629 | Subjects: 19 | Sessions: 0 | Runs: 19
Subjects: ['01', '02', '04', '05', '06', '07', '08', '09', '10', '11', '12', '14', '15', '16', '17', '18', '19', '20', '21']
Sessions: []
Data types: ['meg']
[]
Tasks: ['MMNHCS', 'noise']


{'Name': 'SINGSING',
 'BIDSVersion': '1.7.0',
 'License': 'CC0',
 'DatasetType': 'raw',
 'Authors': ['Valerie Chanoine',
  'Jean-Michel Badier',
  'Mireille Besson',
  'Talya Inbar'],
 'Acknowledgements': 'MEG data acquisition was performed in the MEG Centre (Timone Hospital, Marseille, France)',
 'Funding': ['This research has been supported by funding from the Institute of Convergence ILCB (France 2030, ANR-16-CONV-0002) and the Excellence Initiative of Aix-Marseille University A*MIDEX (ANR-11-IDEX-0001-02)'],
 'ReferencesAndLinks': ['a data paper',
  'a resource to be cited when using the data'],
 'DatasetDOI': 'doi:10.18112/openneuro.ds006629.v1.0.1',
 'GeneratedBy': [{'Name': 'MNE-BIDS',
   'Version': '0.14',
   'Description': 'MNE-BIDS is a Python package that allows you to read and write BIDS-compatible datasets with the help of MNE-Python.'}],
 'SourceDatasets': [{'DOI': 'doi:10.18112/openneuro.ds006629.v1.0.0',
   'URL': 'https://openneuro.org/datasets/ds006629',
   'Version':

In [ ]:
# Load configs
import os
import yaml
from nipype import config as nconfig

config_path = "config/config_effort.yaml"
with open(config_path, "r") as yamlfile:
    config = yaml.load(yamlfile, Loader=yaml.FullLoader)

wf_config = config['workflow']
paths_config = config['paths']

# Configure Nipype logging (applies to all subprocesses)
nconfig.update_config({
    'logging': {
        'log_directory': os.path.join(paths_config["workdir"], 'logs'),
        'log_to_file': True,
        'interface_level': 'info',
        'workflow_level': 'info',
    },
    'execution': {
        'crashdump_dir': os.path.abspath('crashes'),
        'remove_unnecessary_outputs': False,
    }
})

# Create workflow
wf = create_meg_preprocessing(
    basedir=paths_config['basedir'], 
    workdir=paths_config['workdir'], 
    output_dir=paths_config['outputdir'],
    file_templates=paths_config.get('file_templates', None),
    iterable_fields=paths_config.get('iterable_fields', None),
    iterable_values=paths_config.get('iterable_values', None),
    pipeline_config=config['pipeline_config']
    )

# visualize workflow graph
wf.write_graph(graph2use='colored', simple_form=True)
print(f"Workflow graph saved to: {wf.base_dir}/megpreproc/graph.png")

# Run workflow
n_workers = wf_config.get("n_workers", max(1, os.cpu_count() - 2))
print(f"Running with {n_workers} workers")

result = wf.run(
    plugin=wf_config["plugin"],
    plugin_args={"n_procs": n_workers}
)

raw_dir /Users/peli/Projects/Repositories/MEGPypes/data/effort
Iterating over fields: ['subject', 'session']
Provided iterable values: {'subject': ['0001', '0002'], 'session': ['01', '02']}
Processing iterable field: subject
Using custom values for field: subject
Processing iterable field: session
Using custom values for field: session
Final iterables dict: {'subject': ['0001', '0002'], 'session': ['01', '02']}
Valid inputs for initial_preproc: {'l_freq', 'max_buffer', 'min_buffer', 'h_freq', 'gradcomp_auto', 'stim_channel', 'trait_added', 'in_file', 'out_file', 'trait_modified', 'gradcomp_order'}
Step 'crop' argument 'stim_channel': None
Step 'crop' argument 'min_buffer': -0.2
Step 'crop' argument 'max_buffer': 0.5
Step 'filter' argument 'l_freq': 1.0
Step 'filter' argument 'h_freq': 100
Step 'gradcomp' argument 'gradcomp_auto': True
Step 'gradcomp' argument 'order': None
Valid inputs for artifact_rejection: {'ica_plot_path', 'out_file', 'enable_zapline', 'n_chunks', 'in_file', 'ica_n

2026-03-13 11:57:48,310 [INFO] megpypes.interfaces.initpreproc: WF: Initial Preproc | In-File: /Users/peli/Projects/Repositories/MEGPypes/data/effort/0001_effortlearning_20250805_01.ds/0001_effortlearning_20250805_01.meg4


ds directory : /Users/peli/Projects/Repositories/MEGPypes/data/effort/0001_effortlearning_20250805_01.ds
    res4 data read.
    hc data read.
    Separate EEG position data file read.
    Quaternion matching (desired vs. transformed):
      -0.30   68.13    0.00 mm <->   -0.30   68.13   -0.00 mm (orig :  -44.05   52.70 -261.58 mm) diff =    0.000 mm
       0.30  -68.13    0.00 mm <->    0.30  -68.13   -0.00 mm (orig :   52.69  -43.26 -262.62 mm) diff =    0.000 mm
      91.72    0.00    0.00 mm <->   91.72   -0.00    0.00 mm (orig :   67.41   67.51 -239.98 mm) diff =    0.000 mm
    Coordinate transformations established.
    Polhemus data for 3 HPI coils added
    Device coordinate locations for 3 HPI coils added
    64 EEG electrode locations assigned to channel info.
    64 EEG locations added to Polhemus data.
    Measurement info composed.
Finding samples for /Users/peli/Projects/Repositories/MEGPypes/data/effort/0001_effortlearning_20250805_01.ds/0001_effortlearning_20250805_01.

2026-03-13 11:57:53,899 [INFO] megpypes.interfaces.initpreproc: OUT FILE PATH: initial_preproc_raw.fif


Writing /Users/peli/Projects/Repositories/MEGPypes/workdir/megpreproc/_session_01_subject_0001/initial_preproc/initial_preproc_raw.fif
Closing /Users/peli/Projects/Repositories/MEGPypes/workdir/megpreproc/_session_01_subject_0001/initial_preproc/initial_preproc_raw.fif
[done]


...

260313-11:58:04,77 nipype.workflow INFO:
	 [Node] Finished "initial_preproc", elapsed time 8.567999s.
260313-11:58:04,81 nipype.workflow INFO:
	 [Node] Setting-up "megpreproc.artifact_rejection" in "/Users/peli/Projects/Repositories/MEGPypes/workdir/megpreproc/_session_01_subject_0001/artifact_rejection".
260313-11:58:04,99 nipype.workflow INFO:
	 [Node] Executing "artifact_rejection" <megpypes.interfaces.artifact_rejection.ArtifactRejection>


2026-03-13 11:58:04,105 [INFO] megpypes.interfaces.artifact_rejection: NODE: Artifact Rejection | In-File: /Users/peli/Projects/Repositories/MEGPypes/workdir/megpreproc/_session_01_subject_0001/initial_preproc/initial_preproc_raw.fif


Opening raw data file /Users/peli/Projects/Repositories/MEGPypes/workdir/megpreproc/_session_01_subject_0001/initial_preproc/initial_preproc_raw.fif...
    Read 5 compensation matrices
    Range : 6996 ... 773425 =      5.970 ...   659.989 secs
Ready.
Current compensation grade : 0
Reading 0 ... 766429  =      0.000 ...   654.019 secs...
Power of components removed by DSS: 0.14
Iteration 0 score: 3.3697109023279074e-29
Power of components removed by DSS: 0.02
Iteration 1 score: -9.771095168901596e-30
Power of components removed by DSS: 0.14
Iteration 0 score: 3.111610197507736e-29
Power of components removed by DSS: 0.02
Iteration 1 score: -9.458438503896574e-30
Power of components removed by DSS: 0.06
Iteration 0 score: 2.5868594890148714e-29
Power of components removed by DSS: 0.01
Iteration 1 score: -9.078189179435935e-30
Power of components removed by DSS: 0.11
Iteration 0 score: 2.370431631535606e-29
Power of components removed by DSS: 0.02
Iteration 1 score: -8.587366257453916e-3

/Users/peli/Projects/Repositories/MEGPypes/src/megpypes/interfaces/artifact_rejection.py:91: RuntimeWarning: This filename (/Users/peli/Projects/Repositories/MEGPypes/workdir/megpreproc/_session_01_subject_0001/artifact_rejection/ica-icasolution.fif) does not conform to MNE naming conventions. All ICA files should end with -ica.fif, -ica.fif.gz, _ica.fif or _ica.fif.gz
  ica_comps.save(ica_path, overwrite=True)
2026-03-13 12:00:01,435 [INFO] megpypes.interfaces.artifact_rejection: Saved ICA: /Users/peli/Projects/Repositories/MEGPypes/workdir/megpreproc/_session_01_subject_0001/artifact_rejection/ica-icasolution.fif
2026-03-13 12:00:02,680 [INFO] megpypes.interfaces.artifact_rejection: OUT FILE PATH: artifact_cleaned_raw.fif


Writing /Users/peli/Projects/Repositories/MEGPypes/workdir/megpreproc/_session_01_subject_0001/artifact_rejection/artifact_cleaned_raw.fif
